In [170]:
import uproot
import matplotlib.pyplot as plt
import pandas as pd
import tqdm
import numpy as np
import os
import glob
import awkward as ak
import boost_histogram as bh
import mplhep as mh
from matplotlib.colors import LogNorm
import math
mh.style.use("ATLAS")
import particle
# !pip install pyarrow
# !pip install "pyarrow<21" --break-system-packages

In [171]:
data_2022_2023 = uproot.open("build/2022_2023.bck.root:eventID_pass", library="pd")
data_2024_noCaloNu = uproot.open("build/2024_noCaloNu.bck.root:eventID_pass", library="pd")
data_2024_CaloNu = uproot.open("build/2024_CaloNu.bck.root:eventID_pass", library="pd")

In [172]:
x = uproot.open("build/200093.bck.root:nt", library="pd")
xa = x.arrays(library="pd")
print(xa)

    BCID   Calo0_E_EM  Calo0_E_dep  Calo0_nMIP   Calo1_E_EM  Calo1_E_dep  \
0      0  2119.290283   375.692383    6.422092  1975.526489   350.206970   
1      0  2616.737305   463.876160    7.929507  1710.838867   303.285065   
2      0  1169.011108   207.233795    3.542458  2675.613525   474.313293   
3      0  1484.082153   263.087280    4.497219  1943.406494   344.512970   
4      0  2643.216797   468.570221    8.009748  1524.610229   270.271820   
5      0  1108.559692   196.517395    3.359272  1962.435913   347.886353   
6      0  2361.649414   418.656036    7.156513  2550.666504   452.163605   
7      0  1846.003174   327.246002    5.593949  1949.770874   345.641205   
8      0  3258.352539   577.617065    9.873796  1829.166138   324.261261   
9      0  1119.883667   198.524826    3.393587  2754.030029   488.214417   
10     0  2820.270996   499.957153    8.546276  1961.222656   347.671295   
11     0  2635.596436   467.219360    7.986656  2339.043945   414.648682   
12     0  27

In [173]:
events_2022_2023 = data_2022_2023.arrays(library="pd")
events_2024_noCaloNu = data_2024_noCaloNu.arrays(library="pd")
events_2024_CaloNu = data_2024_CaloNu.arrays(library="pd")

events_2022_2023["weight"] = 66 / 10_000
events_2024_noCaloNu["weight"] = 50 / 10_000
events_2024_CaloNu["weight"] = 70 / 10_000
events_2025_2026 = events_2024_noCaloNu
events_2025_2026["weight"] = 154 / 10_000

In [174]:
events_all = pd.concat([events_2022_2023, events_2024_noCaloNu, events_2024_CaloNu, events_2025_2026], ignore_index=True)

In [175]:
events_signal = events_all[events_all["truth_isCharmDimuon"] == True]
events_background = events_all[events_all["truth_isCharmDimuon"] == False]

In [176]:
for x in events_signal.columns:
    print(x)

passed_Remove_CaloNu_PMT_region_truth
passed_In_Faser_Nu_Box_or_Lead_Block
passed_Veto20_and_Veto21_charge_gt_40_pC
passed_Timing_Station_Charge_gt_20_pC
passed_Preshower_Charge_gt_2_5_pC
passed_Calo_total_E_EM_lt_11_GeV
passed_At_least_two_long_tracks
passed_Lead_Track_pz_gt_50_GeV
passed_Subleading_Track_pz_gt_30_GeV
passed_Leading_track_has_at_geq_7_layers
passed_Track_nDoF_geq_9
passed_Leading_track_has_chi2_ndof_lt_15
passed_Track_Pair_z_closest_approach_lt_minus2283_mm
passed_Oppositeminussign_tracks
passed_No_more_than_3_tracks
passed_Track_R_at_max_radius_lt_100_mm
passed_VetoNu0_reduced_charge_lt_30_pC
passed_VetoNu1_reduced_charge_lt_30_pC
run
eventID
truth_isCharm
truth_isCharmDimuon
nu_interaction_type
n_matchedMuonTracks
n_charmMatchedMuonTracks
weight


In [181]:
cuts = [
    "passed_Remove_CaloNu_PMT_region_truth",
    "passed_In_Faser_Nu_Box_or_Lead_Block",
    "passed_At_least_two_long_tracks",
    "passed_Veto20_and_Veto21_charge_gt_40_pC",
    "passed_Timing_Station_Charge_gt_20_pC",
    "passed_Preshower_Charge_gt_2_5_pC",
    "passed_Calo_total_E_EM_lt_11_GeV",
    
    "passed_Lead_Track_pz_gt_50_GeV",
    "passed_Subleading_Track_pz_gt_30_GeV",
    "passed_Leading_track_has_at_geq_7_layers",
    "passed_Track_nDoF_geq_9",
    "passed_Leading_track_has_chi2_ndof_lt_15",
    "passed_Track_Pair_z_closest_approach_lt_minus2283_mm",
    "passed_Oppositeminussign_tracks",
    "passed_No_more_than_3_tracks",
    "passed_Track_R_at_max_radius_lt_100_mm",
    "passed_VetoNu0_reduced_charge_lt_30_pC",
    "passed_VetoNu1_reduced_charge_lt_30_pC"
]

In [189]:
def make_cutflow(events, cuts):
    cutflow = {}

    events = events.copy()

    cutflow["No cuts"] = {"All": len(events), "Passed": len(events), "Efficiency": 1.0,  "Cum. Efficiency": 1.0 ,"Weighted (350 /fb)": events["weight"].sum()}

    for cut in cuts:
        all = len(events)
        events = events[events[cut] == True]
        passed = len(events)
        cummaulative_efficiency = passed / cutflow["No cuts"]["All"] if cutflow["No cuts"]["All"] > 0 else 0
        cutflow[cut] = {"All": all, "Passed": passed, "Efficiency": passed / all if all > 0 else 0, "Weighted (350 /fb)": events["weight"].sum(), "Cum. Efficiency": cummaulative_efficiency}

    cutflow = pd.DataFrame(cutflow).T
    return cutflow

In [190]:
cutflow_signal = make_cutflow(events_signal, cuts)
cutflow_background = make_cutflow(events_background, cuts)

In [191]:
print("Signal Cutflow:")
display(cutflow_signal)
print("\nBackground Cutflow:")
display(cutflow_background)

Signal Cutflow:


,All,Passed,Efficiency,Cum. Efficiency,Weighted (350 /fb)
No cuts,2772.0,2772.0,1.000000,1.000000,32.3540
passed_Remove_CaloNu_PMT_region_truth,2772.0,2770.0,0.999278,0.999278,32.3400
passed_In_Faser_Nu_Box_or_Lead_Block,2770.0,2596.0,0.937184,0.936508,30.6444
passed_At_least_two_long_tracks,2596.0,1061.0,0.408706,0.382756,12.2706
passed_Veto20_and_Veto21_charge_gt_40_pC,1061.0,1061.0,1.000000,0.382756,12.2706
passed_Timing_Station_Charge_gt_20_pC,1061.0,1061.0,1.000000,0.382756,12.2706
passed_Preshower_Charge_gt_2_5_pC,1061.0,1047.0,0.986805,0.377706,12.0898
passed_Calo_total_E_EM_lt_11_GeV,1047.0,791.0,0.755492,0.285354,9.2542
passed_Lead_Track_pz_gt_50_GeV,791.0,773.0,0.977244,0.278860,9.0466
passed_Subleading_Track_pz_gt_30_GeV,773.0,581.0,0.751617,0.209596,6.7834



Background Cutflow:


,All,Passed,Efficiency,Cum. Efficiency,Weighted (350 /fb)
No cuts,5657167.0,5657167.0,1.000000,1.000000,63364.0030
passed_Remove_CaloNu_PMT_region_truth,5657167.0,5654561.0,0.999539,0.999539,63345.7610
passed_In_Faser_Nu_Box_or_Lead_Block,5654561.0,2056218.0,0.363639,0.363471,23629.2328
passed_At_least_two_long_tracks,2056218.0,27359.0,0.013305,0.004836,303.9714
passed_Veto20_and_Veto21_charge_gt_40_pC,27359.0,27340.0,0.999306,0.004833,303.7564
passed_Timing_Station_Charge_gt_20_pC,27340.0,27333.0,0.999744,0.004832,303.6746
passed_Preshower_Charge_gt_2_5_pC,27333.0,27174.0,0.994183,0.004803,301.9064
passed_Calo_total_E_EM_lt_11_GeV,27174.0,7369.0,0.271178,0.001303,83.2490
passed_Lead_Track_pz_gt_50_GeV,7369.0,6510.0,0.883431,0.001151,73.6380
passed_Subleading_Track_pz_gt_30_GeV,6510.0,2050.0,0.314900,0.000362,22.9320
